<div dir="rtl">

# 🎙️ رافِدَين — تدريب أقوى نموذج تحويل صوت لحظي على اللهجة العراقية (RunPod)

دفتر **Jupyter** كامل وشامل لضبط نموذج **Seed-VC `seed-uvit-tat-xlsr-tiny`** (25 مليون باراميتر، 22050Hz) على اللهجة العراقية،
انطلاقاً من مشروع [Alqudimi/rafidain](https://github.com/Alqudimi/rafidain) وتشغيله على منصة **RunPod** —
تجربة مثل Google Colab لكن أقوى: GPU أسرع، بيانات عراقية أكبر (حتى **334 ساعة**)، و**حفظ دائم حقيقي** للنموذج.

> **الاستنساخ اللحظي هنا يعني**: تتكلم بأي صوت → النموذج يحوّل صوتك **لحظياً** إلى الصوت العراقي الهدف مع الحفاظ على الكلام واللهجة.
> النموذج tiny هو الأسرع في عائلة Seed-VC (~430ms زمن وصول منشور رسمياً ببروفايل المكالمات) والضبط الدقيق **لا يبطئه إطلاقاً** —
> نفس البنية، لكن بفهم أعمق للفونولوجيا العراقية (الچاف، القلقلة، إيقاع الجملة، المفردات المحلية).

## 🗺️ ماذا يفعل هذا الدفتر؟

| # | المرحلة | التفاصيل |
|---|---------|----------|
| 1 | 🔍 البيئة | فحص GPU والقرص، تثبيت الاعتماديات المقفلة (نفس إصدارات السكربت الرسمي) |
| 2 | 🎙️ البيانات النظيفة | مجموعات يوسف + الشيخ + صفاء (كلام عراقي نظيف) |
| 3 | 📡 بيانات YODAS | جلب محدد بالبايت من أرشيف espnet/yodas3: **49,194 مقطعاً عراقياً مفهرساً** (334 ساعة) مرتبة بالجودة — ننزّل الأفضل أولاً |
| 4 | ✂️ التجهيز | تقطيع ≤29.5 ث وترميز Opus أحادي 24kHz بالمواصفات التي يطلبها `train.py` |
| 5 | 🏋️ التدريب | ضبط دقيق يبدأ من الأوزان المنشورة رسمياً `DiT_uvit_tat_xlsr_ema.pth` (ليس من الصفر!) + استئناف تلقائي + مرآة نقاط للمقارنة |
| 6 | 🎧 التقييم | مقارنة **A/B سماعية** (مشغلات صوت داخل الدفتر) + **موضوعية** (تشابه متحدث CAMPPlus) بين النموذج الأساس والمضبوط |
| 7 | ⚡ السرعة | قياس RTF فعلي على الـ GPU نفسه لإثبات الجاهزية اللحظية |
| 8 | ☁️ الحفظ الدائم | تصدير + **رفع إلى Hugging Face Hub (يبقى للأبد)** + نسخ إلى تخزين RunPod الدائم |

> ### 🔧 مبدأ التصميم: كل شيء داخل الخلايا — بلا سطر أوامر لمشروعك
> هذا الدفتر لا يشغّل ملفات المشروع عبر سطر الأوامر: خط البيانات العراقي يُستورد **كوحدة بايثون
> داخل النواة** (`iraqi.cmd_yodas(...)`)، و`train.py` يُستورد وتُستدعى صنفته `Trainer` مباشرة،
> و`inference.py` تُستورد وتُستدعى دالتها `main()` — أسلوب Colab الأصيل: تقدم حي في مخرجات الخلية
> نفسها، وأخطاء بايثون حقيقية بتتبعها الكامل، وذاكرة واحدة مشتركة. (أدوات النظام فقط مثل
> `git` و`ffmpeg` تبقى استدعاءات نظام لأنها ليست من ملفات مشروعك).

## ⏱️ الأزمنة التقديرية (RTX 4090)

| المرحلة | الزمن | ملاحظات |
|---------|-------|---------|
| الاعتماديات + النقاط | ~10 دقائق | PyTorch موجود مسبقاً في قوالب RunPod |
| تنزيل 200 ساعة | ~15-30 دقيقة | ~13GB جلب محدد + 1.4GB مجموعات نظيفة |
| التجهيز | ~15-30 دقيقة | 8 عمّال ffmpeg متوازيين |
| **التدريب 4000 خطوة** | **~1.5-3 ساعات** | يعتمد على الـ GPU (T4: ~5 ساعات) |
| التقييم + التصدير + الحفظ | ~15 دقيقة | |

**الحجم الكلي على القرص: ~40-55GB** (لـ 200 ساعة / 334 ساعة).

## ✅ ضمانات هذا الدفتر

1. **كل خلية مكتوبة ومدققة** بناءً على تحليل كامل للكود المنبع (rafidain + seed-vc) سطراً بسطر —
   بما فيه تنفيذ خط البيانات العراقي على الفهرس الحقيقي (49,194 مقطعاً) للتأكد من سلامته قبل اعتماده،
   مع شبكة أمان تتحقق من أهم سطر فيه عند كل تشغيل.
2. **كل مرحلة قابلة للاستئناف**: انقطع الإنترنت أو أوقفت الـ Pod؟ أعد تشغيل الخلايا من حيث توقفت.
3. **الحفظ الدائم بثلاث طبقات** أهمها Hugging Face Hub — نموذجك يبقى حتى لو حُذف الـ Pod نهائياً.
4. **التقييم قبل الادعاء**: مقارنة موضوعية رقمية + سماعية A/B بين الأساس والمضبوط — الأرقام أمامك.

> ⚠️ **ملاحظة صدق مهنية**: لا يستطيع أحد بضمير أن «يضمن أن هذا أفضل نموذج على الإطلاق» قبل قياسه.
> ما نضمنه: أقوى وصفة تدريب موثقة للمشروع + أكبر بيانات عراقية متاحة + تقييم يثبت التحسن بالأرقام. وإن أردت الدفع further —
> آخر خلية فيها خارطة طريق (334 ساعة كاملة، خطوات أكثر، ضبط متحدث محدد فوق نموذج اللهجة).

</div>

<div dir="rtl">

## 🖥️ خطوة بخطوة: تجهيز الـ Pod على RunPod (مرة واحدة)

1. ادخل إلى **[console.runpod.io](https://console.runpod.io)** → تبويب **Pods** → زر **Deploy** (Deploy On-Demand Cloud GPU).
2. **اختر الـ GPU** — الترتيب حسب القيمة مقابل السعر:
   - 🥇 **RTX 4090** (24GB) — الأفضل سعراً/سرعة لهذا التدريب
   - 🥈 **A40 / RTX A6000** (48GB) — دفعة أكبر (batch 6) وتدريب أسرع
   - 🥉 **RTX 3090** (24GB) — بديل اقتصادي
   - ⚠️ T4 (16GB) يعمل لكنه الأبطأ (batch 2 فقط)
3. **اختر القالب (Template)**: **RunPod PyTorch 2.4.0** (مثل `runpod/pytorch:2.4.0-py3.11-cuda12.4.1`) —
   أي قالب PyTorch رسمي حديث يعمل؛ هذا الدفتر يكتشف ما هو موجود ويكمل الناقص تلقائياً.
4. **الإعدادات الحرجة** (Edit Disk Configuration):
   - **Container Disk**: `30` GB
   - **Volume Disk**: `100` GB ← ⭐ يُحمَّل على `/workspace` ويصمد لإيقاف/إعادة تشغيل الـ Pod
5. **المنافذ**: `8888/http` (موجود افتراضياً في قوالب PyTorch — هذا منفذ JupyterLab).
   أضف متغير بيئة `JUPYTER_PASSWORD` بكلمة سر قوية إذاً طلبها القالب.
6. اضغط **Deploy** وانتظر حتى يصبح الـ Pod **Running**.
7. اضغط **Connect** → ضمن **HTTP Services** افتح **Jupyter Lab**.
8. ارفع هذا الدفتر بزر الرفع ⬆️ في متصفح ملفات JupyterLab (يفضل وضعه في `/workspace/`) ثم افتحه ونفّذ الخلايا **بالترتيب من الأعلى للأسفل** (Shift+Enter).

### 💡 قواعد ذهبية أثناء التشغيل
- **إيقاف مؤقت (Stop)** للـ Pod يحفظ `/workspace` كاملاً — البيانات والنقاط والتدريب تبقى.
- **الحذف النهائي (Terminate)** يمسح كل شيء إلا ما رُفع إلى Hugging Face — لذلك ضع التوكن في خلية الإعدادات.
- انقطعت الكهرباء/الإنترنت بالمنتصف؟ لا مشكلة: أعد تشغيل الـ Pod ونفّذ الخلايا بالترتيب — كل مرحلة تكمل نفسها تلقائياً.
- خلية التدريب مقاطعتها **آمنة**: التدريب يستمر بخلفية الـ Pod، وإعادة تشغيل الخلية تلتصق به من جديد.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️ الإعدادات الرئيسية — عدّل القيم هنا فقط ثم نفّذ بالترتيب
# ═══════════════════════════════════════════════════════════════

# ---------------------- التدريب ----------------------
YODAS_HOURS      = "200"    # حجم بيانات YODAS العراقية: "200" (موصى بها، ~13GB) | "all" = 334 ساعة (~22GB) | "50" | "0" لتعطيلها
IRAQI_STEPS      = 4000     # خطوات التدريب — 1500-4000 هي الجرعة الموثقة في FINETUNING.md لتكيّف اللهجة
IRAQI_SAVE_EVERY = 250      # حفظ نقطة تفتيش كل N خطوة (تُنسخ كلها لمرآة المقارنة A/B)
RUN_NAME         = "iraqi_v1"
USE_SMALL_SETS   = True     # مجموعات الكلام العراقي النظيف: يوسف + الشيخ + صفاء (~1.4GB)
FETCH_WORKERS    = 8        # خيوط جلب YODAS المتوازية (8-16 مناسب لشبكة RunPod السريعة)
PREP_WORKERS     = 0        # 0 = تلقائي (نوى المعالج، بحد أقصى 8)
BATCH_SIZE       = 0        # 0 = تلقائي حسب ذاكرة GPU: 16GB→2 | 24GB→4 | 48GB→6 | 80GB→8
TRAIN_NUM_WORKERS = 2       # عمّال تحميل بيانات التدريب (خفّضها إلى 0 إن ظهرت أي مشكلة توازي)
FORCE_RETRAIN    = False    # True = تابع/أعد التدريب حتى لو وُجد نموذج نهائي (مفيد بعد رفع IRAQI_STEPS)

# ------------------- الحفظ الدائم -------------------
HF_TOKEN   = ""             # 🔑 توكن hf_... من https://huggingface.co/settings/tokens (صلاحية Write)
                            #    ⭐ ضعه لضمان بقاء نموذجك للأبد حتى لو حُذف الـ Pod نهائياً
HF_REPO_ID = ""             # مثل "username/rafidain-seedvc-tiny-iraqi" — يُنشأ تلقائياً إن تركته فارغاً
HF_PRIVATE = True           # مستودع خاص على Hugging Face (لا يراه أحد غيرك)

# ------------------- المسارات (RunPod) -------------------
DATA_ROOT = "/workspace/iraqi_seedvc"   # على قرص الحجم الدائم — يصمد لإيقاف وإعادة تشغيل الـ Pod

# ═════════════════ لا تعدّل ما يلي ═════════════════
import os, sys

SEEDVC        = os.path.join(DATA_ROOT, "seed-vc")
RAFIDAIN_SRC  = os.path.join(DATA_ROOT, "rafidain-src")
PIPE          = os.path.join(DATA_ROOT, "iraqi_pipeline.py")
DATA_DIR      = os.path.join(DATA_ROOT, "data", "iraqi_finetune")
RUN_DIR       = os.path.join(SEEDVC, "runs", RUN_NAME)
CFG           = os.path.join(SEEDVC, "configs", "presets", "config_dit_mel_seed_uvit_xlsr_tiny.yml")
FT_MODEL      = os.path.join(RUN_DIR, "ft_model.pth")
CKPT_MIRROR   = os.path.join(DATA_ROOT, "ckpt_mirror", RUN_NAME)
EXPORT_PARENT = os.path.join(DATA_ROOT, "export")
for _p in (DATA_ROOT, EXPORT_PARENT):
    os.makedirs(_p, exist_ok=True)

# متغيرات بيئة حرجة (نفس إعدادات السكربتات الرسمية للمشروع)
os.environ["HF_HUB_DISABLE_XET"] = "1"                # تنزيل HTTP قابل للاستئناف بدل Xet (يتجنب حدود المعدل)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # تحميل نقاط .pth الكلاسيكية (إلزامي لـ torch>=2.6)
os.environ["PYTHONUNBUFFERED"] = "1"                  # بث مخرجات التدريب فورياً


print("✓ الإعدادات جاهزة — كل المراحل التالية استدعاءات مباشرة داخل الخلايا (بلا سطر أوامر)")
print(f"   بيانات YODAS     : {YODAS_HOURS} ساعة" + (" (كامل الأرشيف العراقي)" if YODAS_HOURS == "all" else ""))
print(f"   خطوات التدريب    : {IRAQI_STEPS} خطوة (حفظ كل {IRAQI_SAVE_EVERY})")
print(f"   جذر البيانات     : {DATA_ROOT}")
print(f"   مجلد التدريب     : {RUN_DIR}")
print(f"   الحفظ الدائم HF  : " + ("✅ مفعّل" if HF_TOKEN else "⚠️  معطّل — ضع HF_TOKEN لضمان بقاء النموذج للأبد"))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔍 فحص البيئة: GPU / القرص / الذاكرة — وتحديد حجم الدفعة تلقائياً
# ═══════════════════════════════════════════════════════════════
import subprocess, shutil

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

# ---------------------- GPU ----------------------
gpu_name, vram_gb = "غير مكتشف", 0
_r = _run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"])
if _r.returncode == 0 and _r.stdout.strip():
    _parts = _r.stdout.strip().splitlines()[0].split(",")
    gpu_name = _parts[0].strip()
    vram_gb = int(float(_parts[1])) // 1024
else:
    print("❌ لا يوجد GPU! هذا الدفتر يتطلب Pod بمعالج NVIDIA GPU.")
    print("   أنشئ Pod بقالب RunPod PyTorch الرسمي مع GPU (مثل RTX 4090) ثم أعد تشغيل الخلايا.")
    raise RuntimeError("No NVIDIA GPU detected")

# حجم الدفعة التلقائي وفق وصفة المشروع (2 لكل ~16GB ذاكرة)
if BATCH_SIZE == 0:
    BATCH = 8 if vram_gb >= 70 else 6 if vram_gb >= 40 else 4 if vram_gb >= 20 else 2
else:
    BATCH = int(BATCH_SIZE)

# ------------------- القرص والذاكرة -------------------
_du = shutil.disk_usage(DATA_ROOT)
free_gb = _du.free / 1e9
_yodas_gb = 22.3 if YODAS_HOURS == "all" else (0.0 if str(YODAS_HOURS) == "0" else float(YODAS_HOURS) / 334.4 * 22.3)
need_gb = _yodas_gb + 20.0   # مجموعات نظيفة 1.4 + نقاط 2.3 + بيانات مجهزة + مرآة نقاط التدريب + هوامش

try:
    ram_gb = int(_run(["free", "-g"]).stdout.splitlines()[1].split()[1])
except Exception:
    ram_gb = 0
cpus = os.cpu_count() or 4
PREP_WORKERS_AUTO = min(8, cpus) if PREP_WORKERS == 0 else int(PREP_WORKERS)
FETCH_WORKERS_EFF = max(1, min(int(FETCH_WORKERS), 16))

print("═" * 56)
print(f"   GPU              : {gpu_name}  ({vram_gb} GB VRAM)")
print(f"   حجم الدفعة       : {BATCH}   (وصفة FINETUNING.md: 2 لكل 16GB)")
print(f"   المساحة الحرة    : {free_gb:.0f} GB في {DATA_ROOT}")
print(f"   المطلوب تقديرياً : ~{need_gb:.0f} GB (بيانات + نقاط + مرآة)")
print(f"   CPU / RAM        : {cpus} نواة / {ram_gb} GB")
print(f"   عمّال التجهيز    : {PREP_WORKERS_AUTO}  |  خيوط الجلب: {FETCH_WORKERS_EFF}")
print("═" * 56)

if free_gb < need_gb:
    print(f"⚠️  المساحة الحرة ({free_gb:.0f}GB) أقل من المطلوب تقديرياً (~{need_gb:.0f}GB).")
    print("   الحلول: كبّر Volume Disk من إعدادات الـ Pod، أو خفّض YODAS_HOURS،")
    print("   أو نفّذ التدريب على مراحل (التدريب يستأنف تلقائياً).")
else:
    print("✓ البيئة جاهزة — انتقل للخلية التالية")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 اعتمادات النظام: ffmpeg (التقطيع/الترميز/الفحص) و git
# (قوالب RunPod الرسمية تحتويها غالباً — نتحقق ونكمل الناقص)
# ═══════════════════════════════════════════════════════════════
import shutil, subprocess

_missing = [c for c in ("ffmpeg", "ffprobe", "git") if shutil.which(c) is None]
if _missing:
    print(f"⏳ تثبيت: {', '.join(_missing)} ...")
    _r = subprocess.run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + " ".join(_missing),
                        shell=True, capture_output=True, text=True)
    if _r.returncode != 0:
        print(_r.stdout[-600:])
        print(_r.stderr[-600:])
        raise RuntimeError("فشل تثبيت الاعتمادات — نفّذ يدوياً في طرفية: apt-get install -y " + " ".join(_missing))
else:
    print("✓ ffmpeg / ffprobe / git موجودة مسبقاً")

for _c in ("ffmpeg", "git"):
    _v = subprocess.run([_c, "-version"], capture_output=True, text=True).stdout.splitlines()
    print("  ", _v[0][:78] if _v else _c)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🐍 اعتمادات بايثون — تُثبَّت داخل نواة الدفتر مباشرة
# الإصدارات المقفلة هي نفسها إصدارات السكربت الرسمي seed_vc_iraqi_ubuntu.sh
# ═══════════════════════════════════════════════════════════════
import importlib, importlib.util, subprocess, sys

def pip_install(pkgs):
    print("⏳ pip install:", " ".join(pkgs[:5]), "..." if len(pkgs) > 5 else "", flush=True)
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        print(_r.stdout[-1200:])
        print(_r.stderr[-1200:])
        raise RuntimeError("فشل pip — انسخ الخطأ أعلاه وابحث عنه، أو أعد تشغيل الخلية")
    print("   ✓ تم", flush=True)

# 1) PyTorch: قوالب RunPod الرسمية تأتي بـ torch متوافق مع CUDA — نستعمله كما هو
_have_torch = importlib.util.find_spec("torch") is not None
if _have_torch:
    import torch
    print(f"✓ PyTorch {torch.__version__} موجود مسبقاً — CUDA متاح: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        print("   ⚠️ torch بلا CUDA — سيُعاد تثبيته بإصدار GPU")
        _have_torch = False
if not _have_torch:
    pip_install(["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0",
                 "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("   (إن ظهر خطأ استيراد لاحقاً بعد هذا التثبيت: Kernel → Restart ثم أعد الخلايا بالترتيب)")

# 2) الاعتماديات المقفلة (numpy 1.26 إلزامي مع torch 2.4 + transformers 4.46)
PINNED = [
    "numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "soundfile==0.12.1",
    "munch==4.0.0", "einops==0.8.0", "transformers==4.46.3",
    "pyyaml", "tqdm", "requests", "pyarrow==17.0.0", "huggingface_hub",
    "soxr", "matplotlib", "pandas",
]
pip_install(PINNED)

# 3) تحقق نهائي بأن كل شيء يستورد من نواة الدفتر
for _mod in ("torch", "numpy", "scipy", "librosa", "soundfile", "munch", "einops",
             "transformers", "yaml", "tqdm", "requests", "pyarrow",
             "huggingface_hub", "soxr", "matplotlib", "pandas"):
    importlib.import_module(_mod)

import torch
print("═" * 56)
print(f"✓ كل الاعتماديات جاهزة داخل النواة")
print(f"   torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")

<div dir="rtl">

### 📥 استنساخ المستودعات واستيراد خط البيانات العراقي داخل النواة

نستنسخ **seed-vc** (كود التدريب) و **rafidain** (مشروعك)، ثم نستخرج خط البيانات المضمّن في
`seed_vc_iraqi_ubuntu.sh` — ويحتوي **فهرساً عراقياً كاملاً**: 49,194 مقطعاً مصنّفاً (334 ساعة)
مع خريطة جلب محددة بالبايت من `espnet/yodas3` وترتيب جودة لكل مصدر.

> ### 🔧 كل شيء داخل الخلايا — بلا سطر أوامر لمشروعك
> في هذا الدفتر لا نستدعي ملفات المشروع عبر CLI: خط البيانات يُستورد **كوحدة بايثون داخل النواة**
> (`iraqi.cmd_yodas(...)` بدل `python iraqi_pipeline.py yodas ...`)، والتدريب والاستدلال كذلك
> (استيراد `train.py` و`inference.py` واستدعاء دوالها مباشرة). الفوائد: أشرطة تقدم حية في مخرجات
> الخلية نفسها، أخطاء بايثون حقيقية بتتبعها الكامل، ولا انفصال بين الخلية والعملية.
> (أدوات النظام فقط مثل `git` و`ffmpeg` تبقى استدعاءات نظام — لأنها ليست من ملفات مشروعك).

> ### 🔬 تدقيق عميق + شبكة أمان لخط البيانات
> أثناء تحليل المستودع دُقّق السطر الأهم — دمج نوافذ المقاطع العراقية — **حرفياً ونُفّذ فعلياً على
> بيانات الفهرس الحقيقية** (100 مصدر ← 2,518 مهمة تدريب صحيحة). هذا السطر لو اعتلّ لانهارت مرحلة
> تجهيز YODAS كاملة، وفحص `ast` وحده لا يكشفه لأن الخلل يكون وقت التشغيل — لذلك تتحقق الخلية
> منه حرفياً عند كل تشغيل وتستبدله بالنسخة الصحيحة إن صادفت إصداراً معطوباً (شبكة أمان ضد أي
> انحدار مستقبلي من المنبع).

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📥 استنساخ + استخراج خط البيانات + استيراده داخل النواة مباشرة
# ═══════════════════════════════════════════════════════════════
import subprocess, os, ast, sys, importlib.util, gc

def git_clone(url, dest):
    if os.path.isdir(os.path.join(dest, ".git")):
        print(f"✓ موجود مسبقاً (تخطي الاستنساخ): {dest}")
        return
    print(f"⏳ استنساخ {url} ...")
    _r = subprocess.run(["git", "clone", "--depth", "1", url, dest],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("فشل الاستنساخ: " + _r.stderr[-400:])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN_SRC)

# ⭐ seed-vc يستخدم مسارات نسبية (./runs و./checkpoints) → نضبط دليل العمل مرة واحدة
os.chdir(SEEDVC)
if SEEDVC not in sys.path:
    sys.path.insert(0, SEEDVC)

# ---------- استخراج خط البيانات المضمّن (بالـ manifests العراقية) ----------
_sh = open(os.path.join(RAFIDAIN_SRC, "seed_vc_iraqi_ubuntu.sh"), encoding="utf-8").read()
_marker = "cat > \"$ROOT/iraqi_pipeline.py\" <<'PYEOF'"
_start = _sh.index(_marker) + len(_marker) + 1
_end = _sh.index("\nPYEOF", _start)
pipeline_py = _sh[_start:_end]

# ---------- 🔬 تحقق سلامة دمج نوافذ YODAS (+ شبكة أمان) ----------
CORRECT = 'merged.append([max(0.0, c["st"]), c["en"]])'
if CORRECT in pipeline_py:
    print("✓ دمج نوافذ YODAS سليم في هذه النسخة (دُقّق حرفياً ونُفّذ على بيانات حقيقية أثناء التحليل)")
else:
    _lines = pipeline_py.split("\n")
    _hits = 0
    for _i, _l in enumerate(_lines):
        if "merged.append" in _l and 'c["st"]' in _l:
            _indent = _l[:len(_l) - len(_l.lstrip())]
            _lines[_i] = _indent + CORRECT
            _hits += 1
    pipeline_py = "\n".join(_lines)
    if _hits:
        print(f"🛠️  استُبدل نمط دمج غير مطابق بالنسخة الصحيحة ({_hits} موضع) — شبكة الأمان عملت")
    else:
        print("⚠️  لم يُعثر على سطر الدمج — تابع وراقب مرحلة التجهيز لأي خطأ")

ast.parse(pipeline_py)
with open(PIPE, "w", encoding="utf-8") as _f:
    _f.write(pipeline_py)

# ---------- ⭐ استيراد خط البيانات كوحدة داخل النواة (بلا سطر أوامر) ----------
_spec = importlib.util.spec_from_file_location("iraqi_pipeline", PIPE)
iraqi = importlib.util.module_from_spec(_spec)
sys.modules["iraqi_pipeline"] = iraqi
_spec.loader.exec_module(iraqi)   # يفكّ الفهرس العراقي المضمّن (49,194 مقطعاً)
print(f"✓ وحدة خط البيانات مستوردة داخل النواة: {PIPE}")

# ---------- تحقق الفهرس العراقي (قراءة مباشرة من الوحدة المستوردة) ----------
_resolvable = len([1 for v in iraqi.OFFS.values() if v.get("off") is not None])
print(f"   مقاطع عراقية مفهرسة : {len(iraqi.CLIPS):,}")
print(f"   إجمالي الساعات      : {round(iraqi.TOTAL_CLIP_HOURS, 1)}")
print(f"   مصادر قابلة للجلب   : {_resolvable}")

# ---------- فحص تشغيلي حي: توليد مهام التقطيع لأول مصدر متعدد المقاطع ----------
class _FakeArgs:
    out = "/tmp/iraqi_prep_probe"   # التوليد لا يكتب ملفات — يبني قائمة المهام فقط
_probe_sid = next((sid for sid, cl in iraqi.CLIPS_BY_SID.items()
                   if len(cl) >= 2 and iraqi.OFFS.get(sid, {}).get("off") is not None), None)
if _probe_sid:
    _jobs = list(iraqi._prep_job_src(_FakeArgs, os.path.join("/tmp", f"{_probe_sid}.webm"), "yodas"))
    print(f"✓ فحص تشغيلي: المصدر {_probe_sid[:10]}… → {len(_jobs)} مهمة تقطيع صالحة (الكود يعمل فعلاً)")

# ---------- ملفات seed-vc الأساسية ----------
for _f in ("train.py", "inference.py",
           "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"):
    assert os.path.exists(os.path.join(SEEDVC, _f)), f"ملف مفقود في seed-vc: {_f}"
print("✓ seed-vc سليم: train.py + inference.py + الإعدادات الرسمية")

# ═══════════════════════════════════════════════════════════════
# مساعدات مشتركة للخلايا التالية (استدعاء مباشر — بلا سطر أوامر)
# ═══════════════════════════════════════════════════════════════
_INF_CACHE = {}

def get_inference_mod():
    """استيراد inference.py كوحدة داخل النواة (يُحمَّل مرة واحدة ويُعاد استخدامه)"""
    if os.getcwd() != SEEDVC:
        os.chdir(SEEDVC)
    if SEEDVC not in sys.path:
        sys.path.insert(0, SEEDVC)
    if "mod" not in _INF_CACHE:
        _sp = importlib.util.spec_from_file_location("seedvc_inference",
                                                     os.path.join(SEEDVC, "inference.py"))
        _mod = importlib.util.module_from_spec(_sp)
        sys.modules["seedvc_inference"] = _mod
        _sp.loader.exec_module(_mod)
        _INF_CACHE["mod"] = _mod
    return _INF_CACHE["mod"]

def convert_inprocess(ckpt, cfg, src, tgt, outdir, steps=25):
    """تحويل صوتي بالاستدعاء المباشر لـ inference.main() داخل النواة.
    ملاحظة عدالة A/B: نمرر النقطة والإعدادات صراحةً دائماً — لأن الافتراضي في
    inference.py هو نموذج آخر كلياً (seed-v2 whisper) وليس النموذج tiny مدار التدريب."""
    from types import SimpleNamespace
    import glob as _glob, torch
    inf = get_inference_mod()
    os.makedirs(outdir, exist_ok=True)
    ns = SimpleNamespace(
        source=os.path.abspath(src), target=os.path.abspath(tgt),
        output=os.path.abspath(outdir),
        diffusion_steps=int(steps), length_adjust=1.0, inference_cfg_rate=0.7,
        f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
        checkpoint=os.path.abspath(ckpt), config=os.path.abspath(cfg),
        fp16=True,
    )
    inf.main(ns)
    gc.collect(); torch.cuda.empty_cache()
    _wavs = _glob.glob(os.path.join(outdir, "*.wav"))
    assert _wavs, "لم يُنتج الاستدلال ملفاً"
    return _wavs[0]

def train_busy():
    """هل التدريب ما زال يعمل في خيط خلفي داخل النواة؟"""
    import threading
    _st = globals().get("TRAIN_STATE")
    return (_st is not None
            and isinstance(_st.get("thread"), threading.Thread)
            and _st["thread"].is_alive())

print("✓ المساعدات جاهزة: iraqi.* + convert_inprocess() + train_busy()")

<div dir="rtl">

### 🎙️ البيانات 1/3 — المجموعات النظيفة (يوسف / الشيخ / صفاء)

| المجموعة | المصدر على Hugging Face | الوصف |
|---|---|---|
| **youssef** | `xtz999/iraqi_arabic_speech_youssef` | 3,867 مقطعاً عراقياً @22.05kHz مع نصوص (~720MB) |
| **shiekh** | `karimBD/Iraqi_shiekh_hara_sawtarabi_MediaSpeech` | 827 مقطعاً لشيخ عراقي @16kHz (~640MB) |
| **safaa** | `safaa94/Myvoice-iraqi-accent-samples` | عينات صوتية باللهجة العراقية |

⭐ استدعاء مباشر داخل الخلية: `iraqi.cmd_small(...)` — أشرطة التقدم تظهر في مخرجات الخلية نفسها،
والمرحلة قابلة للاستئناف (علامات `.done` تتخطى ما اكتمل).

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎙️ تنزيل المجموعات النظيفة — استدعاء مباشر لـ cmd_small داخل النواة
# ═══════════════════════════════════════════════════════════════
import os
from types import SimpleNamespace

if USE_SMALL_SETS:
    print("تنزيل المجموعات النظيفة: يوسف (~720MB) + الشيخ (~640MB) + صفاء ...")
    iraqi.cmd_small(SimpleNamespace(root=DATA_ROOT, only=None))
    print()
    for _name in ("youssef", "shiekh", "safaa"):
        _d = os.path.join(DATA_ROOT, "raw", _name)
        _cnt = len(os.listdir(_d)) if os.path.isdir(_d) else 0
        _sz = sum(os.path.getsize(os.path.join(_d, f)) for f in os.listdir(_d)) / 1e6 if _cnt else 0
        print(f"   {_name:8s}: {_cnt:5d} ملف  ({_sz:.0f} MB)")
    print("✓ المجموعات النظيفة جاهزة")
else:
    print("⏭️  تخطي المجموعات الصغيرة (USE_SMALL_SETS=False)")

<div dir="rtl">

### 📡 البيانات 2/3 — جلب الكلام العراقي من أرشيف YODAS (القلب الرئيسي)

**جلب محدد بالبايت (HTTP Range)** من أرشيف `espnet/yodas3` — بدل تنزيل مئات الغيغابايت،
ننزّل **فقط** التسجيلات المصدر التي فيها مقاطع عراقية مصنّفة، مرتبةً بجودة الصوت (الأفضل أولاً):

- خيارك `YODAS_HOURS`: `"200"` → ~1,605 تسجيلات ≈ **12.7GB** (الموصى به) | `"all"` → **334 ساعة / 22.3GB** كاملة
- ⭐ استدعاء مباشر: `iraqi.cmd_yodas(...)` — 8 خيوط جلب متوازية، واستئناف كامل بعد أي انقطاع

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📡 جلب الكلام العراقي من espnet/yodas3 — استدعاء مباشر لـ cmd_yodas
# ═══════════════════════════════════════════════════════════════
import os
from types import SimpleNamespace

if str(YODAS_HOURS) == "0":
    print("⏭️  تخطي YODAS (YODAS_HOURS=0)")
else:
    _label = "كامل الأرشيف (334 ساعة)" if str(YODAS_HOURS).lower() == "all" else f"{YODAS_HOURS} ساعة"
    print(f"جلب {_label} من الكلام العراقي المفهرس (الأفضل جودةً أولاً) ...")
    print("   قابل للاستئناف: عند أي انقطاع أعد تشغيل هذه الخلية فقط")
    iraqi.cmd_yodas(SimpleNamespace(root=DATA_ROOT, hours=str(YODAS_HOURS),
                                    workers=FETCH_WORKERS_EFF, limit=None))
    _d = os.path.join(DATA_ROOT, "raw", "yodas_webm")
    _files = [f for f in os.listdir(_d) if f.endswith(".webm")] if os.path.isdir(_d) else []
    _gb = sum(os.path.getsize(os.path.join(_d, f)) for f in _files) / 1e9
    print(f"✓ {len(_files)} تسجيلاً مصدراً ({_gb:.1f} GB) جاهزة للتجهيز")

<div dir="rtl">

### ✂️ البيانات 3/3 — التقطيع والترميز إلى صيغة التدريب

كل تسجيل خام يُقصّ إلى مقاطع **≤ 29.5 ثانية** (الحد المقبول في `train.py` هو 1–30 ث) ويُرمَّز
**Opus أحادي 24kHz** عبر ffmpeg بمعالجة متوازية. ⭐ استدعاء مباشر: `iraqi.cmd_prep(...)` —
والنتيجة فهرس `_prep_index.jsonl` + تقرير `_stats.json` بالساعات لكل مصدر.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✂️ تقطيع وترميز كل البيانات الخام — استدعاء مباشر لـ cmd_prep
# ═══════════════════════════════════════════════════════════════
import json, os
from types import SimpleNamespace

print(f"تجهيز البيانات ({PREP_WORKERS_AUTO} عامل ffmpeg متوازٍ) — قد يستغرق 15-40 دقيقة ...")
iraqi.cmd_prep(SimpleNamespace(root=DATA_ROOT, out=DATA_DIR,
                               workers=PREP_WORKERS_AUTO, skip_yodas=False, limit=None))

_stats_path = os.path.join(DATA_DIR, "_stats.json")
if os.path.exists(_stats_path):
    _stats = json.load(open(_stats_path, encoding="utf-8"))
    print("═" * 48)
    for _src_name, _v in _stats["per_source"].items():
        print(f"   {_src_name:10s}: {_v['files']:6d} مقطع  |  {_v['hours']:7.2f} ساعة")
    print("─" * 48)
    print(f"   {'المجموع':10s}: {_stats['total_files']:6d} مقطع  |  {_stats['total_hours']:7.2f} ساعة")
    print("═" * 48)
    assert _stats["total_files"] >= 50, "مقاطع قليلة جداً — راجع مخرجات خلايا التنزيل أعلاه"
    print("✓ بيانات التدريب العراقية جاهزة")

<div dir="rtl">

### ⬇️ تنزيل نقاط النماذج مسبقاً (قبل التدريب)

التدريب يبدأ من الأوزان المنشورة رسمياً ويعتمد مكونات جاهزة — ننزّلها كلها الآن عبر
⭐ استدعاء مباشر `iraqi.cmd_prefetch(...)` حتى لا يتوقف التدريب بالمنتصف:

| المكوّن | المستودع | الدور |
|---|---|---|
| `DiT_uvit_tat_xlsr_ema.pth` | Plachta/Seed-VC | **الأوزان الأساس** التي يبدأ منها الضبط الدقيق |
| `facebook/wav2vec2-xls-r-300m` | snapshot كامل (~1.27GB) | مشفّر المحتوى (128 لغة منها العربية) |
| `campplus_cn_common.bin` | funasr/campplus | مستخرج بصمة المتحدث (شرط الأسلوب) |
| `converter/*` | myshell-ai/OpenVoiceV2 | محوّل لون الصوت + قاعدة بصمات `se_db.pt` |
| `hift.pt` | FunAudioLLM/CosyVoice-300M | الفوكودر (توليد الموجة النهائية) |

> ℹ️ نموذج `rmvpe` (مستخرج F0) **غير مطلوب** هنا: إعدادات النموذج tiny تعمل بـ `f0_condition: false`
> فلا يُبنى مستخرج F0 في التدريب ولا في الاستدلال — تحققنا من ذلك في كود `train.py` و`inference.py`.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⬇️ تنزيل كل نقاط النماذج — استدعاء مباشر لـ cmd_prefetch داخل النواة
# ═══════════════════════════════════════════════════════════════
import os
from types import SimpleNamespace

print("تنزيل النقاط: DiT-tiny / XLSR-300m / CAMPPlus / OpenVoice / HiFT ...")
iraqi.cmd_prefetch(SimpleNamespace(repo=SEEDVC))
print()
print("✓ كل النقاط جاهزة — ابدأ التدريب في الخلية التالية")

<div dir="rtl">

### 🏋️ التدريب — داخل نواة الدفتر مباشرة (استيراد، لا سطر أوامر)

⭐ هذه الخلية **تستورد `train.py` كوحدة بايثون داخل النواة** (الاستيراد آمن: دالة `main`
محمية بـ `if __name__ == "__main__"`)، تبني كائن `Trainer` الرسمي بنفس معاملات CLI،
وتستدعي `trainer.train()` في **خيط خلفي داخل النواة**. ما يعنيه ذلك:

1. **البداية من الأوزان المنشورة** `DiT_uvit_tat_xlsr_ema.pth` — ضبط دقيق لا تدريب من الصفر،
   فيحافظ النموذج على خاصية zero-shot ويتخصص للعراقي (وصفة `FINETUNING.md`).
2. **استئناف تلقائي**: كائن Trainer نفسه يبحث عن آخر `DiT_epoch_*_step_*.pth` في `runs/<run_name>/`
   ويكمل منها (بما فيها حالة المحسّن ورقم الخطوة).
3. **تقدم حي من ذاكرة النواة**: الخلية تقرأ `trainer.iters` و`trainer.epoch` و`trainer.ema_loss`
   مباشرة من الكائن — بلا تحليل سجلات ولا ملفات وسيطة.
4. **مرآة نقاط**: كل `DiT_epoch_*_step_*.pth` تُنسخ إلى `ckpt_mirror/` (train.py يبقي آخر 2 فقط).

> 🛡️ **المقاطعة**: زر Interrupt يوقف الخلية فقط — **خيط التدريب يستمر داخل النواة**، أعد تشغيل
> الخلية لتلتصق به وتتابع. انقطاع المتصفح كذلك لا يضر (النواة حية).
> ⚠️ **إعادة تشغيل النواة (Restart) أو إيقاف الـ Pod** يُنهي خيط التدريب — أعاد التشغيل واستأنف
> من آخر نقطة محفوظة تلقائياً (كل `IRAQI_SAVE_EVERY` خطوة). هذا هو ثمن العمل داخل النواة،
> والتقاطع مع الاستئناف يجعل الخسارة صفرية عملياً.
> ⚠️ لا تشغّل خلايا التقييم أثناء التدريب (تعارض ذاكرة GPU) — الخلايا التالية تتحقق تلقائياً.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🏋️ التدريب داخل النواة: استيراد train.py + خيط خلفي + متابعة حية
# ═══════════════════════════════════════════════════════════════
import importlib.util, sys, os, threading, traceback, time, glob, shutil, json, gc

# ⭐ استيراد وحدة التدريب داخل النواة (لا يشغّل argparse بسبب حماية __main__)
os.chdir(SEEDVC)   # train.py يستخدم مسارات نسبية: ./runs و./checkpoints
if SEEDVC not in sys.path:
    sys.path.insert(0, SEEDVC)
_tspec = importlib.util.spec_from_file_location("seedvc_train", os.path.join(SEEDVC, "train.py"))
train_mod = importlib.util.module_from_spec(_tspec)
sys.modules["seedvc_train"] = train_mod
_tspec.loader.exec_module(train_mod)
os.environ["HF_HUB_CACHE"] = os.path.join(SEEDVC, "checkpoints", "hf_cache")
print(f"✓ وحدة التدريب مستوردة داخل النواة (Trainer جاهز للبناء)")

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(CKPT_MIRROR, exist_ok=True)

if "TRAIN_STATE" not in globals():
    TRAIN_STATE = {"thread": None, "trainer": None, "error": None, "history": []}

def mirror_checkpoints():
    """نسخ أي نقطة جديدة إلى المرآة (train.py يحذف القديمة ويبقي آخر 2 فقط)"""
    for _ck in glob.glob(os.path.join(RUN_DIR, "DiT_epoch_*.pth")):
        _dst = os.path.join(CKPT_MIRROR, os.path.basename(_ck))
        if not os.path.exists(_dst):
            shutil.copy2(_ck, _dst)

def _train_worker():
    """عملية التدريب كاملة داخل خيط النواة — بنفس ما تفعله main() في train.py"""
    try:
        trainer = train_mod.Trainer(
            config_path=CFG,
            pretrained_ckpt_path=None,   # تلقائي: آخر نقطة في runs/<run_name> وإلا الأوزان المنشورة
            data_dir=DATA_DIR,
            run_name=RUN_NAME,
            batch_size=BATCH,
            num_workers=TRAIN_NUM_WORKERS,
            steps=IRAQI_STEPS,
            save_interval=IRAQI_SAVE_EVERY,
            max_epochs=1000,
            device="cuda:0",
        )
        TRAIN_STATE["trainer"] = trainer
        trainer.train()          # يكتب ft_model.pth عند الاكتمال
        TRAIN_STATE["error"] = None
    except Exception:
        TRAIN_STATE["error"] = traceback.format_exc()
    finally:
        import torch
        try:
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass

mirror_checkpoints()
_th = TRAIN_STATE["thread"]

if isinstance(_th, threading.Thread) and _th.is_alive():
    print(f"🔗 خيط التدريب يعمل داخل النواة — الالتصاق به ومتابعة التقدم ...")
elif os.path.exists(FT_MODEL) and not FORCE_RETRAIN:
    _th = None
    TRAIN_STATE["error"] = None
    print(f"✓ النموذج النهائي موجود مسبقاً: {FT_MODEL}")
    print(f"   ({os.path.getsize(FT_MODEL)/1e6:.0f} MB — للتدريب مجدداً أو رفع الخطوات: FORCE_RETRAIN=True)")
else:
    TRAIN_STATE.update({"trainer": None, "error": None})
    _init_step = TRAIN_STATE["history"][-1][1] if TRAIN_STATE["history"] else 0
    print(f"🚀 بدء التدريب داخل النواة: {IRAQI_STEPS} خطوة | دفعة {BATCH} | استئناف من الخطوة {_init_step}")
    print(f"   المخرجات: {RUN_DIR}")
    _th = threading.Thread(target=_train_worker, name="iraqi-train", daemon=True)
    TRAIN_STATE["thread"] = _th
    _th.start()
    time.sleep(3)

if _th is not None:
    from tqdm.auto import tqdm
    _last_it = -1
    _last_mirror = 0.0
    _t0 = time.time()
    pbar = tqdm(total=IRAQI_STEPS, initial=min(TRAIN_STATE["history"][-1][1] if TRAIN_STATE["history"] else 0, IRAQI_STEPS),
                desc="training", unit=" step", dynamic_ncols=True)
    try:
        while _th.is_alive():
            _tr = TRAIN_STATE.get("trainer")
            if _tr is not None:
                _it, _ep, _ls = _tr.iters, _tr.epoch, _tr.ema_loss
                if _it != _last_it:
                    pbar.n = min(_it, IRAQI_STEPS)
                    pbar.set_postfix(epoch=_ep, loss=f"{_ls:.4f}")
                    TRAIN_STATE["history"].append((_ep, _it, _ls))
                    _last_it = _it
            time.sleep(2)
            if time.time() - _last_mirror > 30:
                mirror_checkpoints()
                _last_mirror = time.time()
    except KeyboardInterrupt:
        pbar.close()
        print(f"\n⚠️  قُطعت الخلية — خيط التدريب مستمر داخل النواة")
        print("    أعد تشغيل هذه الخلية للالتصاق به. لا تشغّل التقييم قبل اكتماله.")
        raise
    pbar.close()
    mirror_checkpoints()
    print(f"انتهى خيط التدريب ({(time.time()-_t0)/60:.1f} دقيقة من هذه الجلسة)")

# ------------------ النتيجة ------------------
if TRAIN_STATE.get("error"):
    print("❌ خطأ داخل خيط التدريب:")
    print("\n".join("   " + l for l in TRAIN_STATE["error"].splitlines()[-14:]))
    print("\n   الحل الشائع: نفاد ذاكرة GPU → خفّض BATCH_SIZE في الإعدادات، أو بيانات ناقصة →")
    print("   أعد خلايا التنزيل. ثم أعد هذه الخلية — الاستئناف تلقائي من آخر نقطة.")
elif os.path.exists(FT_MODEL):
    _hist_path = os.path.join(RUN_DIR, "notebook_loss_history.json")
    with open(_hist_path, "w", encoding="utf-8") as _f:
        json.dump(TRAIN_STATE["history"], _f)
    print("═" * 56)
    print("✅ اكتمل التدريب — النموذج النهائي جاهز")
    print(f"   {FT_MODEL}  ({os.path.getsize(FT_MODEL)/1e6:.0f} MB)")
    _mir = glob.glob(os.path.join(CKPT_MIRROR, "DiT_epoch_*.pth"))
    print(f"   نقاط المرآة للمقارنة A/B: {len(_mir)} نقطة في {CKPT_MIRROR}")
    if len(TRAIN_STATE["history"]) >= 3:
        import matplotlib.pyplot as plt
        _steps_h = [h[1] for h in TRAIN_STATE["history"]]
        _losses_h = [h[2] for h in TRAIN_STATE["history"]]
        plt.figure(figsize=(9, 3.5))
        plt.plot(_steps_h, _losses_h, lw=0.9, alpha=0.6, label="EMA loss (sampled live)")
        if len(_losses_h) > 20:
            import numpy as _np
            _k = max(3, len(_losses_h) // 40)
            plt.plot(_steps_h[_k-1:], _np.convolve(_losses_h, _np.ones(_k)/_k, mode="valid"),
                     lw=1.8, label=f"EMA loss (smoothed, k={_k})")
        plt.xlabel("training step"); plt.ylabel("EMA loss")
        plt.title("Training loss — Iraqi dialect fine-tune (read live from Trainer object)")
        plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        plt.show()
    print("\n➡️  التالي: خلية الاختبار السماعي A/B")
else:
    print("❌ لم يُنتج التدريب نموذجاً نهائياً — أعد تشغيل هذه الخلية (الاستئناف تلقائي)")
    print(f"   آخر نقطة مرآة: {CKPT_MIRROR}")

<div dir="rtl">

### 🎧 اختبار سماعي A/B — هل النموذج المضبوط عراقياً أفضل فعلاً؟

نحوّل نفس المصدر بنفس الصوت المرجعي مرتين: مرة بالنموذج **الأساس** ومرة بالنموذج **المضبوط عراقياً** —
⭐ عبر استدعاء مباشر لـ `inference.main()` داخل النواة (الوحدة تُحمَّل مرة وتُعاد استخدامها).

> 🔬 **ملاحظة عدالة مقارنة مهمة**: عند عدم تمرير نقطة، يستخدم `inference.py` افتراضياً نموذجاً
> **آخر كلياً** (seed-v2 whisper). لذلك نمرر صراحةً النقطة الأساس `DiT_uvit_tat_xlsr_ema.pth`
> (نفس ما بدأ منه التدريب) — مقارنة تفاح بتفاح حقيقية.

- **اختبار اللهجة**: مصدر عراقي ← مرجع عراقي — هل النبرة العراقية أكثر طبيعية بعد الضبط؟
- **اختبار zero-shot**: مصدر إنجليزي ← نفس المرجع — إثبات أن التعميم على أي صوت لم يُفقد.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎧 اختبار سماعي A/B داخل النواة — أساس (نقطة tiny الرسمية) مقابل مضبوط
# ═══════════════════════════════════════════════════════════════
import subprocess, os, glob
from IPython.display import Audio, display, HTML

assert "convert_inprocess" in globals(), "شغّل الخلايا بالترتيب (خلايا 1-7 أولاً)"
assert os.path.exists(FT_MODEL), "لا يوجد ft_model.pth — أكمل خلية التدريب أولاً"
assert not train_busy(), "التدريب ما زال يعمل في الخلفية — انتظر اكتماله ثم أعد هذه الخلية"

# النقطة الأساس: نفس أوزان tiny المنشورة التي بدأ منها التدريب (تُقرأ من الكاش فوراً)
from huggingface_hub import hf_hub_download
BASE_CKPT = hf_hub_download(repo_id="Plachta/Seed-VC", filename="DiT_uvit_tat_xlsr_ema.pth",
                            repo_type="model", cache_dir=os.path.join(SEEDVC, "checkpoints"))
print(f"✓ النقطة الأساس (نفس بداية التدريب): {BASE_CKPT}")

_cands = sorted(glob.glob(os.path.join(DATA_DIR, "yodas", "*.ogg")), key=os.path.getsize, reverse=True)
if len(_cands) < 4:
    _cands = sorted(glob.glob(os.path.join(DATA_DIR, "*", "*.ogg")), key=os.path.getsize, reverse=True)
assert len(_cands) >= 4, "لا توجد مقاطع عراقية مجهزة كافية للاختبار"

_tmp = os.path.join(DATA_ROOT, "ab_test")
os.makedirs(_tmp, exist_ok=True)

def to_wav(path):
    """تحويل نسقي عبر ffmpeg (أداة نظام) لعرض واستدلال موحدين"""
    out = os.path.join(_tmp, os.path.splitext(os.path.basename(path))[0] + ".wav")
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path, "-ar", "22050", "-ac", "1", out],
                   check=True)
    return out

ref_voice = to_wav(_cands[0])      # الصوت العراقي الهدف (أطول مقطع)
src_iraqi = to_wav(_cands[3])      # مصدر عراقي مختلف
src_en = os.path.join(SEEDVC, "examples", "source", "source_s1.wav")

def show(title, path):
    display(HTML(f"<h4 style='direction:rtl; margin-bottom:4px'>{title}</h4>"))
    display(Audio(filename=path))

TESTS = [
    ("🇮🇶 اختبار اللهجة: مصدر عراقي ← صوت عراقي هدف", src_iraqi),
    ("🌍 اختبار zero-shot: مصدر إنجليزي ← نفس الصوت العراقي", src_en),
]
print("تشغيل 4 تحويلات داخل النواة (قد يستغرق 2-4 دقائق) ...\n")
for _title, _src in TESTS:
    display(HTML(f"<h3 style='direction:rtl'>{_title}</h3>"))
    _tag = os.path.splitext(os.path.basename(_src))[0][:24]
    out_base = convert_inprocess(BASE_CKPT, CFG, _src, ref_voice, os.path.join(_tmp, f"base_{_tag}"))
    out_ft   = convert_inprocess(FT_MODEL, CFG, _src, ref_voice, os.path.join(_tmp, f"ft_{_tag}"))
    show("🔊 المصدر (الصوت الأصلي الداخل)", _src)
    show("🎯 المرجع (الصوت العراقي المطلوب)", ref_voice)
    show("🔻 مخرج النموذج الأساس (قبل الضبط)", out_base)
    show("🟢 مخرج النموذج المضبوط عراقياً (بعد الضبط)", out_ft)
print("💡 استمع وقرر: هل النبرة العراقية أكثر طبيعية في المخرج الأخضر؟")

<div dir="rtl">

### 📊 تقييم موضوعي A/B — تشابه المتحدث (CAMPPlus cosine)

«السمع ملك» لكن الأرقام تحسم. نقيس بنفس مقاييس `evaluation/evaluate_conversion.py` في مشروعك
ومعايير القبول في `FINETUNING.md` — والتحويلات ⭐ داخل النواة مباشرة:

- **sim(المحوّل، المرجع) يرتفع** ← المخرج يشبه الصوت العراقي الهدف أكثر ✅
- **sim(المحوّل، المصدر) ينخفض** ← أثر صوت المتحدث الأصلي يضعف ✅

4 أزواج (مصدر ومرجع من متحدثين عراقيين مختلفين) + جدول + رسم بياني + حكم تلقائي.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 تقييم موضوعي داخل النواة: CAMPPlus — أساس مقابل مضبوط
# ═══════════════════════════════════════════════════════════════
import subprocess, os, glob, sys

assert "convert_inprocess" in globals(), "شغّل الخلايا بالترتيب (خلايا 1-7 أولاً)"
assert os.path.exists(FT_MODEL), "لا يوجد ft_model.pth — أكمل خلية التدريب أولاً"
assert not train_busy(), "التدريب ما زال يعمل — انتظر اكتماله ثم أعد هذه الخلية"

from huggingface_hub import hf_hub_download
BASE_CKPT = hf_hub_download(repo_id="Plachta/Seed-VC", filename="DiT_uvit_tat_xlsr_ema.pth",
                            repo_type="model", cache_dir=os.path.join(SEEDVC, "checkpoints"))

sys.path.insert(0, SEEDVC)
import numpy as np, torch, torchaudio, librosa
from modules.campplus.DTDNN import CAMPPlus

_ck = hf_hub_download(repo_id="funasr/campplus", filename="campplus_cn_common.bin",
                      repo_type="model", cache_dir=os.path.join(SEEDVC, "checkpoints"))
_spk = CAMPPlus(feat_dim=80, embedding_size=192)
_spk.load_state_dict(torch.load(_ck, map_location="cpu"))
_spk.eval()
_kaldi = torchaudio.compliance.kaldi

def embed(path):
    _wav, _ = librosa.load(path, sr=16000)
    _t = torch.from_numpy(_wav).reshape(1, -1)
    _f = _kaldi.fbank(_t, num_mel_bins=80, dither=0, sample_frequency=16000)
    _f = _f - _f.mean(dim=0, keepdim=True)
    with torch.no_grad():
        return _spk(_f.unsqueeze(0)).squeeze(0).numpy()

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

_cands = sorted(glob.glob(os.path.join(DATA_DIR, "yodas", "*.ogg")), key=os.path.getsize, reverse=True)
if len(_cands) < 8:
    _cands = sorted(glob.glob(os.path.join(DATA_DIR, "*", "*.ogg")), key=os.path.getsize, reverse=True)
assert len(_cands) >= 8, "لا تتوفر مقاطع كافية للتقييم"

_tmp = os.path.join(DATA_ROOT, "eval_obj")
os.makedirs(_tmp, exist_ok=True)

def to_wav(path):
    out = os.path.join(_tmp, os.path.splitext(os.path.basename(path))[0] + ".wav")
    if not os.path.exists(out):
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path, "-ar", "22050", "-ac", "1", out],
                       check=True)
    return out

PAIRS = [(to_wav(_cands[i]), to_wav(_cands[i + 40])) for i in (0, 8, 16, 24)]
rows = []
print("تشغيل 8 تحويلات داخل النواة (أساس + مضبوط لكل زوج) — بضع دقائق ...")
for _i, (_src, _ref) in enumerate(PAIRS, 1):
    _tag = f"pair{_i}"
    out_base = convert_inprocess(BASE_CKPT, CFG, _src, _ref, os.path.join(_tmp, f"base_{_tag}"))
    out_ft   = convert_inprocess(FT_MODEL, CFG, _src, _ref, os.path.join(_tmp, f"ft_{_tag}"))
    e_src, e_ref = embed(_src), embed(_ref)
    e_b, e_f = embed(out_base), embed(out_ft)
    rows.append({
        "test": _tag,
        "sim_ref_base": round(cos(e_b, e_ref), 4),
        "sim_ref_ft":   round(cos(e_f, e_ref), 4),
        "sim_src_base": round(cos(e_b, e_src), 4),
        "sim_src_ft":   round(cos(e_f, e_src), 4),
    })
    print(f"   زوج {_i}/4 ✓  تشابه المرجع: أساس {rows[-1]['sim_ref_base']:.3f} ← مضبوط {rows[-1]['sim_ref_ft']:.3f}")

import pandas as pd
df = pd.DataFrame(rows)
mean_row = {"test": "MEAN",
            "sim_ref_base": round(df.sim_ref_base.mean(), 4),
            "sim_ref_ft":   round(df.sim_ref_ft.mean(), 4),
            "sim_src_base": round(df.sim_src_base.mean(), 4),
            "sim_src_ft":   round(df.sim_src_ft.mean(), 4)}
df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
display(df)

import matplotlib.pyplot as plt
_labels = ["similarity to Iraqi reference\n(higher = better)",
           "similarity to source speaker\n(lower = better)"]
_base_v = [mean_row["sim_ref_base"], mean_row["sim_src_base"]]
_ft_v   = [mean_row["sim_ref_ft"],   mean_row["sim_src_ft"]]
fig, ax = plt.subplots(figsize=(7.5, 4))
_xs = np.arange(2)
ax.bar(_xs - 0.17, _base_v, 0.34, label="base model", color="#9aa0a6")
ax.bar(_xs + 0.17, _ft_v,   0.34, label="Iraqi fine-tuned", color="#2e7d32")
ax.set_xticks(_xs); ax.set_xticklabels(_labels)
ax.set_ylim(0, 1.0)
ax.set_ylabel("CAMPPlus cosine similarity")
ax.set_title("A/B: base vs Iraqi fine-tuned (mean of 4 Iraqi pairs)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for _xi, (_b, _f) in enumerate(zip(_base_v, _ft_v)):
    ax.text(_xi - 0.17, _b + 0.015, f"{_b:.3f}", ha="center", fontsize=9)
    ax.text(_xi + 0.17, _f + 0.015, f"{_f:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

print("═" * 60)
if mean_row["sim_ref_ft"] > mean_row["sim_ref_base"] and mean_row["sim_src_ft"] <= mean_row["sim_src_base"] + 0.02:
    print("✅ النموذج المضبوط أفضل موضوعياً: تشابه أعلى بالمرجع العراقي وأثر أقل للمصدر")
    print(f"   تحسن التشابه بالمرجع: {100*(mean_row['sim_ref_ft']-mean_row['sim_ref_base']):+.2f} نقطة")
elif mean_row["sim_ref_ft"] > mean_row["sim_ref_base"]:
    print(f"🟡 تشابه المرجع تحسن ({100*(mean_row['sim_ref_ft']-mean_row['sim_ref_base']):+.2f}) لكن أثر المصدر لم ينخفض")
    print("   جرّب رفع الخطوات: FORCE_RETRAIN=True مع IRAQI_STEPS أعلى ثم أعد التدريب والتقييم")
else:
    print("🟡 الفرق غير حاسم على هذه العينة — خارطة الدفع للجودة:")
    print("   1) IRAQI_STEPS=6000 + FORCE_RETRAIN=True ثم أعد خلية التدريب والتقييم")
    print("   2) أو YODAS_HOURS='all' (334 ساعة) وأعد من خلية الجلب")
    print("   3) أو قارن نقاط مرآة مبكرة: أحياناً الخطوات المتأخرة تفرط بالتخصيص")
print("═" * 60)
print("💡 معايير FINETUNING.md: تشابه مرجع ↑ + تشابه مصدر ↓ + استماع بشري — أكملناها كلها في هذه الخلية والسابقة")

<div dir="rtl">

### ⚡ قياس السرعة — هل هو جاهز فعلاً للاستنساخ اللحظي؟

النموذج `tiny` هو الأسرع في عائلة Seed-VC، والضبط الدقيق **لا يغيّر سرعته إطلاقاً**
(نفس البنية، نفس خطوات الاستدلال). لكن السرعة تُقاس ولا تُفترض — نقيس **RTF** فعلياً
على الـ GPU نفسه ⭐ بتحويلات داخل النواة (استدعاء `inference.main()` مباشرة):

- تحويل مقطعين بأطوال مختلفة بنفس النقطة وتحليل \( T = L + r \times d \)
  حيث \(L\) زمن تحميل النماذج الثابت و\(r\) زمن المعالجة لكل ثانية صوت — فنستخرج \(r\) بميل الخط.
- **RTF = r** ← أقل من 1.0 يعني أسرع من الزمن الحقيقي على هذا الـ GPU.
- نقيس عند بروفايل المكالمات (`diffusion-steps 10` = إعداد `balanced` في رافدَين) وعند الجودة (`25`).

> مرجع المشروع: زمن الوصول المنشور للنموذج نفسه ببروفايل `balanced` ≈ **430ms** شاملاً الاستدلال —
> خدمة رافدَين تحمّل النموذج مرة واحدة عند الإقلاع، فزمن التحميل أدناه لا يدخل زمن الوصول.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚡ قياس RTF الفعلي داخل النواة للنموذج المضبوط على هذا الـ GPU
# ═══════════════════════════════════════════════════════════════
import subprocess, os, glob, time

assert "convert_inprocess" in globals(), "شغّل الخلايا بالترتيب (خلايا 1-7 أولاً)"
assert os.path.exists(FT_MODEL), "لا يوجد ft_model.pth — أكمل خلية التدريب أولاً"
assert not train_busy(), "التدريب ما زال يعمل — انتظر اكتماله ثم أعد هذه الخلية"

_cands = sorted(glob.glob(os.path.join(DATA_DIR, "yodas", "*.ogg")), key=os.path.getsize, reverse=True)
if len(_cands) < 3:
    _cands = sorted(glob.glob(os.path.join(DATA_DIR, "*", "*.ogg")), key=os.path.getsize, reverse=True)
_tmp = os.path.join(DATA_ROOT, "latency")
os.makedirs(_tmp, exist_ok=True)

def to_wav(path):
    out = os.path.join(_tmp, os.path.splitext(os.path.basename(path))[0] + ".wav")
    if not os.path.exists(out):
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path, "-ar", "22050", "-ac", "1", out],
                       check=True)
    return out

def dur(path):
    _r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                         "-of", "csv=p=0", path], capture_output=True, text=True)
    return float(_r.stdout.strip())

short_clip = to_wav(_cands[2])
long_clip  = to_wav(_cands[0])
ref_voice  = to_wav(_cands[1])
d1, d2 = dur(short_clip), dur(long_clip)
print(f"مقاطع القياس: {d1:.1f}s و {d2:.1f}s (المرجع {dur(ref_voice):.1f}s)")

print()
results = {}
for _steps, _label in ((10, "balanced — 10 خطوات (بروفايل المكالمات في رافدَين)"),
                       (25, "quality — 25 خطوة")):
    _o1 = os.path.join(_tmp, f"out_{_steps}_short")
    t_short = time.perf_counter()
    convert_inprocess(FT_MODEL, CFG, short_clip, ref_voice, _o1, steps=_steps)
    t_short = time.perf_counter() - t_short
    _o2 = os.path.join(_tmp, f"out_{_steps}_long")
    t_long = time.perf_counter()
    convert_inprocess(FT_MODEL, CFG, long_clip, ref_voice, _o2, steps=_steps)
    t_long = time.perf_counter() - t_long
    rtf = (t_long - t_short) / (d2 - d1)
    load_est = t_short - rtf * d1
    results[_steps] = rtf
    _v = "✅ أسرع من الزمن الحقيقي" if rtf < 1.0 else "⚠️ أبطأ من الزمن الحقيقي"
    print(f"   [{_label}]")
    print(f"      RTF = {rtf:.3f}  |  تحميل النماذج ≈ {load_est:.1f}s  →  {_v}")

rtf_bal = results[10]
print("═" * 60)
if rtf_bal < 1.0:
    block_ms = rtf_bal * 180
    print(f"✅ جاهز للاستنساخ اللحظي على {gpu_name}")
    print(f"   RTF = {rtf_bal:.3f} → معالجة كتلة 180ms تستغرق ~{block_ms:.0f}ms على هذا الـ GPU")
    print("   خدمة رافدَين تحمّل النموذج مرة واحدة — زمن الوصول الفعلي في حدود المنشور (~430ms)")
else:
    print(f"⚠️  RTF = {rtf_bal:.3f} — للبث اللحظي استخدم GPU أقوى أو بروفايل ultra_low في الخدمة")
print("═" * 60)

<div dir="rtl">

### 📦 تصدير حزمة النموذج

⭐ استدعاء مباشر داخل النواة: `iraqi.cmd_export(...)` — ينتج مجلداً مكتفياً بذاته:

| الملف | المحتوى |
|---|---|
| `seed_uvit_tat_xlsr_tiny_iraqi.pth` | أوزان النموذج المضبوط عراقياً |
| `config_dit_mel_seed_uvit_xlsr_tiny.yml` | الإعدادات الرسمية (ينسخها من مجلد التدريب تلقائياً — كائن Trainer وضعها هناك عند البدء) |
| `training_report.json` | تقرير كامل: بيانات التدريب بالأرقام (مقاطع/ساعات لكل مصدر) + أصل النموذج + الفهرس العراقي |
| `README.md` | تعليمات الاستخدام والتشغيل |

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 تصدير حزمة النموذج — استدعاء مباشر لـ cmd_export داخل النواة
# ═══════════════════════════════════════════════════════════════
import os
from types import SimpleNamespace

assert os.path.exists(FT_MODEL), "لا يوجد ft_model.pth — أكمل خلية التدريب أولاً"

EXPORT_DIR = os.path.join(EXPORT_PARENT, f"iraqi_seed_vc_tiny_{RUN_NAME}")
iraqi.cmd_export(SimpleNamespace(run_dir=RUN_DIR, config=CFG,
                                 data_dir=DATA_DIR, out=EXPORT_DIR))
assert os.path.isdir(EXPORT_DIR), f"مجلد التصدير غير موجود: {EXPORT_DIR}"

print()
print("═" * 56)
print(f"✓ حزمة النموذج: {EXPORT_DIR}")
for _f in sorted(os.listdir(EXPORT_DIR)):
    _sz = os.path.getsize(os.path.join(EXPORT_DIR, _f)) / 1e6
    print(f"   {_f:45s} {_sz:8.1f} MB")
print("═" * 56)

<div dir="rtl">

### ☁️ الحفظ الدائم — ثلاث طبقات حماية لنموذجك

| الطبقة | المسار | ماذا يصمد أمامه |
|---|---|---|
| 1️⃣ قرص الحجم | `/workspace/...` (نحن هنا أصلاً) | إيقاف/إعادة تشغيل الـ Pod |
| 2️⃣ Network Volume | `/runpod-volume` (إن وُجد) | إيقاف + مسح الحاوية (يبقى ما دمت تدفع تخزينه) |
| 3️⃣ **Hugging Face Hub** | حسابك الخاص | **كل شيء — حتى الحذف النهائي للـ Pod. للأبد** ⭐ |

الخلية تنفّذ الثلاث تلقائياً:
- تنشئ **ZIP** جاهزاً للتنزيل المباشر من متصفح ملفات JupyterLab،
- تنسخه إلى Network Volume إن وجد،
- وترفع الحزمة كاملة إلى مستودعك على Hugging Face (خاص افتراضياً) إن وضعت `HF_TOKEN`.

> 🔑 إن لم تضع التوكن بعد: ضعه في خلية الإعدادات ثم أعد تشغيل **هذه الخلية فقط** — لا حاجة لإعادة أي شيء آخر.

</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ☁️ الحفظ الدائم: ZIP + Network Volume + Hugging Face Hub
# ═══════════════════════════════════════════════════════════════
import os, shutil

assert os.path.isdir(EXPORT_DIR), "شغّل خلية التصدير أولاً"

# ---------- 1) ZIP للتنزيل المباشر من JupyterLab ----------
_zip_base = os.path.join(EXPORT_PARENT, f"iraqi_seed_vc_tiny_{RUN_NAME}")
zip_path = shutil.make_archive(_zip_base, "zip", EXPORT_DIR)
print(f"📦 حزمة ZIP جاهزة ({os.path.getsize(zip_path)/1e6:.0f} MB):")
print(f"   {zip_path}")
print("   (نزّلها من متصفح ملفات JupyterLab بالنقر عليها → Download)")

# ---------- 2) نسخ إلى Network Volume إن وُجد ----------
NV = "/runpod-volume"
if os.path.isdir(NV):
    _dst = os.path.join(NV, "rafidain_exports")
    os.makedirs(_dst, exist_ok=True)
    shutil.copy2(zip_path, _dst)
    shutil.copytree(EXPORT_DIR, os.path.join(_dst, os.path.basename(EXPORT_DIR)),
                    dirs_exist_ok=True)
    print(f"\n✓ نُسخ إلى قرص الشبكة الدائم: {_dst}")
else:
    print("\nℹ️  لا يوجد Network Volume محمّل على /runpod-volume — (اختياري، الطبقة 3 تكفي)")

# ---------- 3) Hugging Face Hub: الأبدية الحقيقية ----------
if HF_TOKEN:
    from huggingface_hub import HfApi, create_repo
    api = HfApi(token=HF_TOKEN)
    if not HF_REPO_ID:
        _who = api.whoami()["name"]
        repo_id = f"{_who}/rafidain-seedvc-tiny-iraqi"
    else:
        repo_id = HF_REPO_ID
    print(f"\n⏳ رفع النموذج إلى Hugging Face: {repo_id} ...")
    create_repo(repo_id, private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=EXPORT_DIR, repo_id=repo_id, repo_type="model",
                      commit_message="Iraqi dialect fine-tune of seed-uvit-tat-xlsr-tiny (rafidain)")
    api.upload_file(path_or_fileobj=zip_path, path_in_repo=os.path.basename(zip_path),
                    repo_id=repo_id, repo_type="model")
    print("═" * 60)
    print("✅ نُشر النموذج — رابط دائم يبقى مهما حدث للـ Pod:")
    print(f"   https://huggingface.co/{repo_id}")
    print()
    print("   للتنزيل مستقبلاً على أي جهاز:")
    print(f"     huggingface-cli download {repo_id} --local-dir ./iraqi_model")
    print("═" * 60)
else:
    print("\n⚠️  الحفظ الأبدي على Hugging Face معطّل (لا توكن)")
    print("   ضع HF_TOKEN في خلية الإعدادات ثم أعد تشغيل هذه الخلية فقط.")
    print(f"   حالياً النموذج محفوظ محلياً: {EXPORT_DIR}")

# ---------- ملخص المواقع ----------
print()
print("📋 خريطة الحفظ النهائية:")
print(f"   • الأوزان الخام + السجل + المرايا : {RUN_DIR}")
print(f"   • حزمة التصدير                    : {EXPORT_DIR}")
print(f"   • ملف ZIP                         : {zip_path}")
if HF_TOKEN:
    print(f"   • Hugging Face (دائم)             : https://huggingface.co/{repo_id}")
print()
print("⚠️ تذكير: كل ما على /workspace يزول مع الحذف النهائي (Terminate) للـ Pod —")
print("   ما يبقى حقاً هو نسخة Hugging Face. اضمنها قبل إيقاف الحساب.")

<div dir="rtl">

### 🚀 النشر والاستخدام بعد التدريب

> الأوامر التالية **خارج الدفتر** — للاستخدام الإنتاجي على خوادم وأجهزة أخرى
> (داخل الدفتر كان كل شيء استدعاءً مباشراً بالاستيراد داخل الخلايا).

#### 1) داخل خدمة رافِدَين (الإنتاج — بث لحظي WebSocket)
```bash
# في .env لخدمة rafidain — النقطة المضبوطة تحل محل تنزيل HuggingFace تلقائياً:
RAFIDAIN__ENGINE__LOCAL_CHECKPOINT=/workspace/iraqi_seedvc/seed-vc/runs/iraqi_v1/ft_model.pth
RAFIDAIN__ENGINE__LOCAL_CONFIG=/workspace/iraqi_seedvc/seed-vc/configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml
RAFIDAIN__ENGINE__DEVICE=cuda
```
التراجع للنموذج الرسمي = حذف السطرين وإعادة التشغيل. مقارنة A/B حية: عقدتان (أساس/مضبوط) ومقارنة المقاييس.

#### 2) استدلال مستقل (أي جهاز فيه GPU)
```bash
cd seed-vc
python inference.py \
  --checkpoint <مسار>/seed_uvit_tat_xlsr_tiny_iraqi.pth \
  --config     <مسار>/config_dit_mel_seed_uvit_xlsr_tiny.yml \
  --source المتحدث.wav --target الصوت_العرقي_الهدف.wav \
  --output ./out --diffusion-steps 25 --fp16 true
```

#### 3) واجهة ويب تفاعلية (Gradio)
```bash
cd seed-vc
python app_vc.py --checkpoint <مسار>/seed_uvit_tat_xlsr_tiny_iraqi.pth \
                 --config <مسار>/config_dit_mel_seed_uvit_xlsr_tiny.yml
```

#### 4) استعادة النموذج على Pod جديد في المستقبل
```bash
huggingface-cli download <username>/rafidain-seedvc-tiny-iraqi --local-dir ./iraqi_model
```

#### 5) مواصلة التدريب لاحقاً (مثلاً بعد جمع بيانات أكثر)
انقل مجلد `runs/<run_name>/` (أو نزّله من HF)، ثم على الـ Pod الجديد: نفس خلايا هذا الدفتر —
التدريب يستأنف تلقائياً من آخر نقطة، ويمكنك رفع `IRAQI_STEPS` ووضع `FORCE_RETRAIN=True`.

</div>

<div dir="rtl">

### ✅ الضمانات وحدودها — بصراحة كاملة

#### ما يضمنه هذا الدفتر (ويمكنك التحقق منه بنفسك)
1. **سلامة التنفيذ**: كل خلية كُتبت بعد تحليل سطر-بسطر للمستودعين (rafidain + seed-vc)،
   وخط البيانات العراقي نُفّذ فعلياً على فهرسه الحقيقي (49,194 مقطعاً ← آلاف مهام التدريب الصحيحة)
   قبل اعتماده، مع شبكة أمان تتحقق من سلامته عند كل تشغيل.
2. **الاستئناف الكامل**: كل مرحلة (تنزيل/تجهيز/تدريب) تكمل نفسها بعد أي انقطاع — بتصميم لا يضيع عملاً.
3. **الحفظ الدائم**: بثلاث طبقات، وأقواها Hugging Face — النموذج يبقى قابل التنزيل بعد سنوات حتى لو حُذف الحساب على RunPod.
4. **التقييم قبل الادعاء**: مقارنة A/B سماعية وموضوعية بالأرقام بين الأساس والمضبوط، وقياس RTF فعلي.
5. **السرعة محفوظة**: الضبط الدقيق لا يمس بنية النموذج ولا خطوات الاستدلال — السرعة قبل التدريب = السرعة بعده (وخلية القياس تثبتها).

#### ما لا يستطيع أحد ضمانه بصدق (وكيف ندفعه للأقصى)
- **«أفضل نموذج عراقي على الإطلاق»** حكم مطلق يحتاج مقارنة بكل النماذج الأخرى وب اختبارات سماع موسعة.
  ما نقدمه بدلاً من الوعد: أقوى وصفة موثقة للمشروع + أكبر بيانات عراقية متاحة علناً + تقييم يثبت التحسن بالأرقام.
- **جودة الصوت الخام** مرتبطة بجودة المرجع الذي تعطيه للنموذج: أعطه 5-30 ثانية صوت عراقي نظيف بلا موسيقى وضجيج.

#### ⬆️ خارطة دفع الجودة أبعد (بترتيب المردود)
| الخطوة | كيف | التكلفة التقريبية |
|---|---|---|
| بيانات أكثر | `YODAS_HOURS="all"` (334 ساعة بدل 200) | +10GB تنزيل، +1-2 ساعة تدريب |
| خطوات أكثر | `IRAQI_STEPS=6000` + `FORCE_RETRAIN=True` ثم خلية التدريب | ساعة-ساعتان إضافيتان |
| نقطة أمثل | قارن نقاط المرآة (خطوة 2000/3000/4000) بنفس تقييم الخلية 📊 — أحياناً نقطة مبكرة تتفوق | دقائق |
| تخصص متحدث واحد | بعد نموذج اللهجة: 5-60 دقيقة لصوت هدف واحد، 200-500 خطوة، run جديد | ~15 دقيقة |
| ضبط الفوكودر HiFT | إن ظهر «طنين/خشونة» رغم تشابه جيد — مسار متقدم خارج نطاق هذا الدفتر | بحث إضافي |

---

<div align="center">

**انتهى الدفتر — نموذجك العراقي الآن في ثلاثة أماكن آمنة** 🇮🇶

</div>

</div>